# SigAlg's `L2.norm` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2.norm` method in SigAlg computes the *$L^2$-norm* of a random variable in the $L^2$-Hilbert space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2.norm).

## Mathematical definition

Let $X \in L^2(\Omega, \mathcal{F}, P)$ be a random variable on a probability space $(\Omega, \mathcal{F}, P)$. The *$L^2$-norm* of $X$ is defined as

$$
\|X\| \stackrel{\text{def}}{=} \sqrt{\langle X, X \rangle} = \sqrt{E(X^2)} = \sqrt{\int_\Omega X^2 \, dP}.
$$

The $L^2$-norm satisfies the following properties:

1. *Positivity*: For all $X \in L^2$,
   $$
   \|X\| \geq 0,
   $$
   with equality if and only if $X = 0$ almost surely.

2. *Homogeneity*: For all $X \in L^2$ and $a \in \mathbb{R}$,
   $$
   \|aX\| = |a| \cdot \|X\|.
   $$

3. *Triangle inequality*: For all $X, Y \in L^2$,
   $$
   \|X + Y\| \leq \|X\| + \|Y\|.
   $$

Together with the inner product, the $L^2$-norm makes $L^2(\Omega, \mathcal{F}, P)$ into a *Hilbert space*.

## API examples


### Basic norms

We begin by setting up a probability space and creating an $L^2$ space.

In [ ]:
from sigalg.core import ProbabilityMeasure, RandomVariable, SampleSpace, SigmaAlgebra
from sigalg.l2 import L2

Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.45,
        3: 0.3,
    }
)

H = L2(sample_space=Omega, sigma_algebra=F, probability_measure=P)

Create a random variable and compute its $L^2$ norm.

In [ ]:
X = RandomVariable(domain=Omega, name="X").from_dict(
    {
        0: 2,
        1: -3,
        2: 2,
        3: -3,
    }
)

norm_X = H.norm(X)
print(f"X: {X}")
print(f"‖X‖ = {norm_X:.6f}")

We can verify that the norm equals $\sqrt{E(X^2)}$.

In [ ]:
import numpy as np

# Compute E(X²) manually
expected_X_squared = sum(X.data[i]**2 * P.data[i] for i in range(4))
sqrt_expected_X_squared = np.sqrt(expected_X_squared)

print(f"E(X²) = {expected_X_squared:.6f}")
print(f"√E(X²) = {sqrt_expected_X_squared:.6f}")
print(f"‖X‖ = {norm_X:.6f}")
print(f"Match: {abs(norm_X - sqrt_expected_X_squared) < 1e-10}")

### Homogeneity

The norm is homogeneous: $\|aX\| = |a| \cdot \|X\|$ for any scalar $a$.

In [ ]:
a = -2.5

norm_aX = H.norm(a * X)
abs_a_times_norm_X = abs(a) * H.norm(X)

print(f"‖{a}X‖ = {norm_aX:.6f}")
print(f"|{a}| · ‖X‖ = {abs_a_times_norm_X:.6f}")
print(f"Homogeneity holds: {abs(norm_aX - abs_a_times_norm_X) < 1e-10}")

### Triangle inequality

The norm satisfies the triangle inequality: $\|X + Y\| \leq \|X\| + \|Y\|$.

In [ ]:
Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: 4,
        2: 1,
        3: 4,
    }
)

norm_X_plus_Y = H.norm(X + Y)
norm_X_plus_norm_Y = H.norm(X) + H.norm(Y)

print(f"‖X + Y‖ = {norm_X_plus_Y:.6f}")
print(f"‖X‖ + ‖Y‖ = {norm_X_plus_norm_Y:.6f}")
print(f"Triangle inequality holds: {norm_X_plus_Y <= norm_X_plus_norm_Y + 1e-10}")

### Norm of orthonormal basis vectors

The orthonormal basis vectors all have norm equal to $1$.

In [ ]:
basis = H.basis
print("Norms of orthonormal basis vectors:")
for atom_id, phi in basis.items():
    norm_phi = H.norm(phi)
    print(f"  ‖φ_{atom_id}‖ = {norm_phi:.10f}")

### Centered vs uncentered norms

The *uncentered norm* $\|X\|$ measures the overall magnitude of $X$, while the *centered norm* $\|X - E(X)\|$ measures the variability around the mean.

In [ ]:
from sigalg.core import Operators

E = Operators.expectation

X.probability_measure = P

# Uncentered norm
norm_uncentered = H.norm(X)

# Centered norm
X_centered = X - E(X)
norm_centered = H.norm(X_centered)

print(f"Uncentered norm ‖X‖ = {norm_uncentered:.6f}")
print(f"Centered norm ‖X - E(X)‖ = {norm_centered:.6f}")

### Connection to standard deviation

The *standard deviation* is the centered $L^2$-norm:

$$
\text{Std}(X) = \|X - E(X)\| = \sqrt{\text{Var}(X)}.
$$

Let's verify this relationship.

In [ ]:
# Using the std method
std_X = Operators.std(X).item()

# Using the norm of centered X
std_X_norm = H.norm(X - E(X))

print(f"Std(X) using std method: {std_X:.6f}")
print(f"Std(X) using norm: {std_X_norm:.6f}")
print(f"Match: {abs(std_X - std_X_norm) < 1e-10}")

We can also verify the relationship with variance: $\text{Std}(X) = \sqrt{\text{Var}(X)}$.

In [ ]:
var_X = Operators.variance(X).item()
sqrt_var_X = np.sqrt(var_X)

print(f"Var(X) = {var_X:.6f}")
print(f"√Var(X) = {sqrt_var_X:.6f}")
print(f"Std(X) = {std_X:.6f}")
print(f"Match: {abs(std_X - sqrt_var_X) < 1e-10}")